# Standalone bubble detector
Use **Godfather**. Explicit page and detector paths are set below; no models.json is required. This notebook detects bubbles only. Use NarrationModelwBubbles_Test.ipynb for OCR filtering, embeddings and the full narration pipeline.


In [ ]:
from pathlib import Path
import hashlib
import json
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

ROOT = Path(r'C:/Users/ESHAAN/BYTE/Applied-Task/1pageTest')
SEQUENCE_ID = "seq_14469aeb38963085"
PAGE_NUMBER = 2
# Dataset page filenames are zero-padded: page 1 is 01.png.
PAGE = ROOT / "TaskData/test/images" / SEQUENCE_ID / f"{PAGE_NUMBER:02d}.png"
# PAGE = ROOT / "TaskData/development/images" / SEQUENCE_ID / f"{PAGE_NUMBER:02d}.png"
OUT = Path(r'C:/Users/ESHAAN/BYTE/Applied-Task/Magi/FINETUNING-EXPERMENTS/NarrationModelwBubbles/cache_selected_ocr/original_notebook_smoke')
OUT.mkdir(parents=True, exist_ok=True)
MODELS = {"bubble_detection": {"local_path": "C:/MODELS/manga_bubbles/bubbles_detect.pt"}}
if not PAGE.is_file():
    raise FileNotFoundError(f"Put your manga page at {PAGE}, or edit PAGE.")
page = Image.open(PAGE).convert("RGB")
PAGE_INFO = {"name": PAGE.name, "sha256": hashlib.sha256(PAGE.read_bytes()).hexdigest(),
             "width": page.width, "height": page.height}

def save_stage(name, records):
    path = OUT / name
    path.write_text(json.dumps({"page": PAGE_INFO, "records": records},
                               ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved {path}")

def load_stage(name):
    path = OUT / name
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}; run the preceding stage first.")
    payload = json.loads(path.read_text(encoding="utf-8"))
    if payload["page"] != PAGE_INFO:
        raise ValueError(f"{name} belongs to a different page. Rerun the earlier stages.")
    return payload["records"]


Run this stage using the settings and helpers above; inspect its printed result before continuing.


In [ ]:
import math
from ultralytics import YOLO

MODEL = Path(MODELS["bubble_detection"]["local_path"])
CONFIDENCE = 0.25
if not MODEL.is_file():
    raise FileNotFoundError(f"Download the bubble detector to {MODEL}; see README.md.")
model = YOLO(str(MODEL))
result = model.predict(source=page, imgsz=1024, conf=CONFIDENCE, verbose=False)[0]
bubbles = []
for box, score in zip(result.boxes.xyxy.cpu().tolist(), result.boxes.conf.cpu().tolist()):
    x1, y1, x2, y2 = box
    bounds = [max(0, math.floor(x1)), max(0, math.floor(y1)),
              min(page.width, math.ceil(x2)), min(page.height, math.ceil(y2))]
    if bounds[2] > bounds[0] and bounds[3] > bounds[1]:
        bubbles.append({"box": bounds, "confidence": float(score)})
bubbles.sort(key=lambda b: (b["box"][1], -b["box"][0]))
for i, bubble in enumerate(bubbles, 1):
    bubble["bubble_id"] = f"B{i}"
save_stage("bubbles.json", bubbles)
print(f"Detected {len(bubbles)} bubbles")


Run this stage using the settings and helpers above; inspect its printed result before continuing.


In [ ]:
preview = page.copy()
draw = ImageDraw.Draw(preview)
for bubble in bubbles:
    x1, y1, x2, y2 = bubble["box"]
    draw.rectangle((x1, y1, x2, y2), outline="blue", width=3)
    draw.text((x1 + 3, y1 + 3), bubble["bubble_id"], fill="blue", stroke_width=1, stroke_fill="white")
preview.save(OUT / "bubble_preview.png")
display(preview)
bubbles
